###  簡単な例



GaussianProcessRegressorは標準ではRBFカーネルを用いますが定義が少し異なります。

$$
K(x,y) = \exp ( - \frac{1}{2\sigma^2} |\!|x-y|\!|_2^2  )
$$

ここでの$\sigma$がコード内でのlength_scaleです。



教材用に乱数の固定をしています。

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor, kernels
from sklearn.gaussian_process.kernels import RBF
from sklearn.preprocessing import StandardScaler
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
%matplotlib inline


In [ ]:
def plot_GPR(X, y, Xtrain, ytrain, yp_mean, yp_std, acq, ia=None, filename=None):
    """plot y.mean += y.std and aquisition functions

    Args:
        X (np.array): descriptor
        y (np.array): target values
        Xtrain (np.array): training descriptor data 
        ytrain (np.array): training target values
        yp_mean (np.array): the mean values of predictions
        yp_std (np.array): the stddev vlaues of predictions
        acq (np.array): aquisition function values
        ia (np.array, optional): a list of actions. Defaults to None.
        filename (str, optional): filename. Defaults to None.
    """
    yminus = yp_mean - yp_std
    yplus = yp_mean + yp_std
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(Xtrain[:, 0], ytrain, "o", color="blue", label="train")
    # alphaで線を半透明にする
    ax.fill_between(X[:, 0], yminus, yplus, color="red", alpha=0.1)
    ax.plot(X[:, 0], yp_mean.reshape(-1),
            color="red", label="predict$\pm\sigma$")
    ax.plot(X[:, 0], y, "--", color="blue", label="expriment")
    ax.plot(X[:, 0], acq, color="green", label="aquisition function")

    if ia is not None:
        ax.axvline(X[ia, 0], color="green", linestyle="--")
        #plt.plot(X[ia,0],yp_mean.reshape(-1)[ia],"o",color="green",label="selected action")
    ax.legend()
    if filename is not None:
        fig.savefig(filename)


In [ ]:
# データ取得
g_df = pd.read_csv("../data_calculated/sin5.csv")
#g_descriptor_names = ['x1']
g_descriptor_names = ['x1', 'x2']
#g_descriptor_names = ['x1','x2','x3' ]
#g_descriptor_names = ['x1','x2','x3','x4']
g_target_name = "y"


In [ ]:
g_Xraw = g_df[g_descriptor_names].values
g_y = g_df[g_target_name].values.reshape(-1)  # 一次元配列に直す。
# データプリプロセス
g_scaler = StandardScaler()
g_X = g_scaler.fit_transform(g_Xraw)


獲得関数としてUCBを用いて一回だけ獲得関数を求めてみる。

In [ ]:
g_seed_initial_selection = 11
g_seed_simulation = 1  # seed for TS
g_v = 0.3



**UCBによる探索**

USBの獲得関数は

$$a_{\textrm{UCB}} = y_\textrm{mean} + k_t  \sigma$$ 

でした。繰り返しが進むほどサンプル点が取られ
$y_\textrm{mean}$は（回帰スコアが良ければ）$y$に近くなっていきます。
一方、繰り返しが進むほど$\sigma$は小さくなっていきます。
小さくなる$\sigma$を$k_t$である程度大きくして未探索点かつ予測値が大きい未探索点を探していきます。


In [ ]:
def search_candidate_UCB(it, train, X, y, reg, v, filename=None):
    """search next action in the UCB method

    Args:
        it (int): the number of iteration
        train (np.array): the index of training data
        X (np.array): descriptor
        y (np.array): target values
        reg (regressor): regressor
        v (float): a factor of sqrt(v*it)*stddev
        filename (str, optional): filename. Defaults to None.

    Returns:
        int: next action
    """
    # GPR training data setの作成
    Xtrain = X[train]
    ytrain = y[train]
    reg.fit(Xtrain, ytrain)
    print("kernel=", reg.kernel_)
    yp_mean, yp_std = reg.predict(X, return_std=True)
    acq = yp_mean + yp_std*np.sqrt(v*it)
    ia = np.argmax(acq)
    plot_GPR(X, y, Xtrain, ytrain, yp_mean, yp_std, acq, ia, filename=filename)
    return ia


def make_model(optimize=True):
    """make a GaussianProcessRegressor with RBF kernel. 

    Args:
        optimize (bool, optional): optimize kernel parameter or not. Defaults to False.

    Returns:
        GaussianProcessRegressor: model.
    """
    if optimize:
        kernel = RBF(length_scale=1)
        reg = GaussianProcessRegressor(kernel=kernel)
    else:
        kernel = RBF(length_scale=1)
        reg = GaussianProcessRegressor(kernel=kernel, optimizer=None)
    return reg


def get_train(nall, nchoice=2, seed_initial_selection=0):
    """take nchoice indexes from from [0:nall-1].

    Args:
        nall (int): sample size.
        nchoice (int, optional): the number of choices. Defaults to 2.
        seed_initial_selection (int, optional): initial seed. Defaults to 0.

    Returns:
        [int]: a list of chosen index.
    """
    # データ解析 (simulationを行う。)
    random.seed(seed_initial_selection)
    idx = range(nall)
    train = random.sample(idx, nchoice)
    return train


g_reg = make_model()

g_train = get_train(g_X.shape[0], 3, g_seed_initial_selection)

print("train", g_train)
os.makedirs("image_executed", exist_ok=True)
for _it in range(20):
    print("\niteration=", _it+1)

    filename = "image_executed/BayseOpt_simple_UCB_{}.png".format(_it)
    g_ia = search_candidate_UCB(
        _it, g_train, g_X, g_y, g_reg, g_v, filename=filename)
    print("next action=", g_ia, "x=", g_X[g_ia, 0])
    g_train = np.hstack([g_train, g_ia])
    print("action=", g_train)


**トンプソンサンプリングによる探索**

また、上のGPRの可視化の仕方だと各サンプル点の分布が独立してあるように見えますが、
実際はyの分布はsample数次元の平均値$\mu$=yp_mean, 共分散行列$\Sigma$=y_covarencematrixの多変量正規分布 $f(x)$ で与えられます。

$$
f(y) = \frac{1}{\sqrt{(2\pi)^n|\Sigma|}} \exp( -\frac{1}{2}(y-\mu)^T \Sigma^{-1} (y-\mu) )
$$
ここに $y$ は $n$ 次元(サンプル数)のベクトルです。

ソースコードの
```
reg.predict(Xw, return_cov=True)
```
が２つめの返り値yp_stdは $\Sigma$ の対角項の平方根です。

* reg.sample_y(X)

もしくは

* scipy.stats.multivariate_normal.rvsもしくはnumpy.random.multivariate_normalで多変量正規分布から一点をランダムに選択できます。

情報理論では分布から一点をランダムに選択することをstochastic samplingといいます。（random samplingと同じです。）

２番めの例を用います。
まずcovarianceを取得します。

In [ ]:
g_train = get_train(g_X.shape[0], 5, g_seed_initial_selection)

g_use_sample_y = True
if g_use_sample_y:
    print("calculate covariance matrix")
    g_Xtrain = g_X[g_train]
    g_ytrain = g_y[g_train]
    g_reg.fit(g_Xtrain, g_ytrain)
    #yp_mean,yp_std = reg.predict(X,return_std=True)
    # covarane matrix(ycov)も計算できる。
    g_yp_mean, g_yp_covarencematrix = g_reg.predict(g_X, return_cov=True)


以下に点線が実験で、実線がmultivariate_normal.rvsによりstochastic samplingで与えられた一連のサンプル点を示します。

In [ ]:
from scipy.stats import multivariate_normal

g_fig, g_ax = plt.subplots()
# 50点選択
for _i in range(50):
    if g_use_sample_y == False:
        g_acq = multivariate_normal.rvs(g_yp_mean, g_yp_covarencematrix)
    else:
        # random_state=0がdefaultなのでrandom_stateを指定しないと全て同じになる。
        g_acq = g_reg.sample_y(g_X, random_state=_i)
    g_ax.plot(g_X[:, 0], g_acq, color="red", alpha=0.1)
    g_ax.plot(g_X[:, 0], g_y, "--", color="blue")  # ,label="expriment")

plt.plot(g_Xtrain[:, 0], g_ytrain, "o", color="blue", label="expriment")


それぞれの線は薄い線で、線が重なった部分は色が濃くなっています。
多数回stochasting samplingを行うとGPRでsigmaを含めて書いた図の赤shadeとほぼ同じなのが分かります。

```python
use_sample_y = True
```
として
reg.sample_yを使用してもほぼ同じ線が得られることも確認してください。


Thompson samplingはy_covarencematrixを用いて次探索点を挙げます。
UCBで必要だったparameter $v$が不要なことが利点です。

In [ ]:
# データ取得

def get_X_y_train(df, descriptor_names, target_name, seed_initial_selection):
    """get normalized X and y from dataframe, and action, which is a list of data of training data.

    Args:
        df (pd.DataFrame): data.
        descriptor_names ([str]]): a list of explanatory variable names.
        target_name (str): target variable name.
        seed_initial_selection (int): initial seed. 

    Returns:
        np.ndarray: X,
        np.ndarray: y,
        [int]: a list of actions.
    """
    Xraw = df[descriptor_names].values
    y = df[target_name].values

    # データプリプロセス
    scaler = StandardScaler()
    X = scaler.fit_transform(Xraw)

    # データ解析 (simulationを行う。)
    random.seed(seed_initial_selection)
    idx = range(X.shape[0])
    train = random.sample(idx, 3)
    return X, y, train


X, y, train = get_X_y_train(
    g_df, g_descriptor_names, g_target_name, g_seed_initial_selection)


def search_candidate_TS(it, train, X, y, reg):
    """search next action in the TS method

    Args:
        it (int): the number of iteration
        train (np.array): the index of training data
        X (np.array): descriptor
        y (np.array): target values
        reg (regressor): regressor

    Returns:
        int: next action
    """
    # GPR training data setの作成
    Xtrain = X[train]
    ytrain = y[train]
    reg.fit(Xtrain, ytrain)
    print("kernel=", reg.kernel_)
    yp_mean, yp_std = reg.predict(X, return_std=True)
    acq = reg.sample_y(X, random_state=it)
    ia = np.argmax(acq)
    plot_GPR(X, y, Xtrain, ytrain, yp_mean, yp_std, acq, ia)
    return ia


random.seed(g_seed_simulation)
reg = GaussianProcessRegressor()
for it in range(8):
    print("iteration=", it+1)
    print("train=", train)
    ia = search_candidate_TS(it, train, X, y, reg)
    print("next action=", ia, "x=", X[ia, 0])
    train = np.hstack([train, ia])


上の探索過程はある乱数での例です。
探索過程は乱数によります。
